# Validation Test Evaluation - Two-phase Couette Flow

This notebook and the corresponding simulation setup notebook (TwoPhaseCouetteFlow_Run.ipynb) are part of published results (cf. 5.3) found in *The extended discontinuous Galerkin method adapted for moving contact line problems via the generalized Navier boundary condition* (https://doi.org/10.1002/fld.5016).

In [ ]:
#r "BoSSSpad.dll"
//#r "..\..\..\src\L4-application\BoSSSpad\bin\Release\net8.0\BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

In [ ]:
wmg.Init("CLpaper_TwoPhaseCouetteFlow");
//OpenOrCreateDatabase(@"\\fdygitrunner\ValidationTests\databases\CLpaper_TwoPhaseCouetteFlow"); 
//OpenOrCreateDatabase(@"\\fdygitrunner\ValidationTests\databases\bkup-2025Oct29_000000.CLpaper_TwoPhaseCouetteFlow");

In [3]:
using System.IO;
using BoSSS.Application.XNSE_Solver.Logging;
using BoSSS.Application.XNSE_Solver.PhysicalBasedTestcases;

## Sessions

In [ ]:
var sessions = wmg.Sessions;
//var sessions = databases.Pick(0).Sessions;
sessions

#0: CLpaper_TwoPhaseCouetteFlow	CouetteGNBC_asymmetric_compDOFstudy_k3_mesh1	5/22/2020 4:08:34 PM	978101ac...
#1: CLpaper_TwoPhaseCouetteFlow	CouetteGNBC_symmetric_compDOFstudy_k3_mesh1	5/22/2020 4:00:56 PM	7bc14576...
#2: CLpaper_TwoPhaseCouetteFlow	CouetteGNBC_symmetric_compDOFstudy_k2_mesh2	5/22/2020 4:01:05 PM	765e6d5e...
#3: CLpaper_TwoPhaseCouetteFlow	CouetteGNBC_symmetric_compDOFstudy_k4_mesh0	5/22/2020 4:00:56 PM	6a7e9d71...
#4: CLpaper_TwoPhaseCouetteFlow	CouetteGNBC_asymmetric_compDOFstudy_k2_mesh2	5/22/2020 4:08:43 PM	5e921ddc...
#5: CLpaper_TwoPhaseCouetteFlow	CouetteGNBC_asymmetric_compDOFstudy_k4_mesh0	5/22/2020 4:08:34 PM	5c8e5ab3...


In [5]:
string[] studyNames = { "k2_mesh2", "k3_mesh1", "k4_mesh0" };
string[] testcase = { "symmetric", "asymmetric" };

In [ ]:
// origin database including all sessions, in case of missing sessions in wmg, due to rerunning only a subset of all needed runs (reduced computational cost)
var originDB = OpenOrCreateDatabase(@"\\fdygitrunner\ValidationTests\databases\bkup-2025Oct29_000000.CLpaper_TwoPhaseCouetteFlow");

In [6]:
List<ISessionInfo> evalSess = new List<ISessionInfo>();

In [ ]:
foreach (string test in testcase) {
foreach (string study in studyNames) {
    string studyName = $"CouetteGNBC_{test}_compDOFstudy_{study}";
    Console.WriteLine("Searching for: " + studyName);
    var studySess = sessions.Where(sess => sess.Name.Contains(studyName));
    int studyCount = studySess.Count();
    if (studyCount == 0) {
        //Console.WriteLine("Not found!");
        
        // try to get from originDB
        var originSess = originDB.Sessions.Where(sess => sess.Name.Contains(studyName));
        int originCount = originSess.Count();
        if (originCount == 0) {
            Console.WriteLine("Not found!");
        } else if (originCount > 1) {
            Console.WriteLine("Restart session found in originDB! Take last run");       
            evalSess.Add(originSess.Where(sess => sess.Name.ToLower().EndsWith($"_restart{originCount-1}")).Single());
        } else {
            evalSess.Add(originSess.Single());
            Console.WriteLine("found in originDB");
        }

    } else if (studyCount > 1) {
        Console.WriteLine("Restart session found! Take last run");       
        evalSess.Add(studySess.Where(sess => sess.Name.ToLower().EndsWith($"_restart{studyCount-1}")).Single());
    } else {
        evalSess.Add(studySess.Single());
        Console.WriteLine("found");
    }
}
}
evalSess

#0: CLpaper_TwoPhaseCouetteFlow	CouetteGNBC_symmetric_compDOFstudy_k2_mesh2	5/22/2020 4:01:05 PM	765e6d5e...
#1: CLpaper_TwoPhaseCouetteFlow	CouetteGNBC_symmetric_compDOFstudy_k3_mesh1	5/22/2020 4:00:56 PM	7bc14576...
#2: CLpaper_TwoPhaseCouetteFlow	CouetteGNBC_symmetric_compDOFstudy_k4_mesh0	5/22/2020 4:00:56 PM	6a7e9d71...
#3: CLpaper_TwoPhaseCouetteFlow	CouetteGNBC_asymmetric_compDOFstudy_k2_mesh2	5/22/2020 4:08:43 PM	5e921ddc...
#4: CLpaper_TwoPhaseCouetteFlow	CouetteGNBC_asymmetric_compDOFstudy_k3_mesh1	5/22/2020 4:08:34 PM	978101ac...
#5: CLpaper_TwoPhaseCouetteFlow	CouetteGNBC_asymmetric_compDOFstudy_k4_mesh0	5/22/2020 4:08:34 PM	5c8e5ab3...


In [8]:
NUnit.Framework.Assert.AreEqual(6, evalSess.Count, $"Not enough sessions for evaluation.");

### compute times

In [9]:
bool calcComputeTimes = false;

In [10]:
if (calcComputeTimes) {

foreach (var sess in evalSess) {
    var nameData = sess.Name.Split(new string[] { "_" }, StringSplitOptions.RemoveEmptyEntries);
    string sessName = "";
    int nameLength = nameData.Length;
    for (int i = 0; i < nameLength; i++) {
        if (nameData[i].StartsWith("restart"))
            continue;
        if (i == 0)
            sessName += nameData[i];
        else    
            sessName += "_" + nameData[i];
    }

    // determine compute time
    Stack<ISessionInfo>  procSIs = new Stack<ISessionInfo>();
    procSIs.Push(sess);
    var currSI = sess;
    var currTimesteps = currSI.Timesteps.OrderBy(ts => ts.TimeStepNumber.MajorNumber);
    int timesteps = currTimesteps.Last().TimeStepNumber.MajorNumber;
    double physicalTime = currTimesteps.Last().PhysicalTime;
    TimeSpan computeTime = currTimesteps.Last().CreationTime - currSI.Timesteps.First().CreationTime;
    var rSIs = sessions.Where(sess => sess.ID.Equals(currSI.RestartedFrom));
    while(!rSIs.IsNullOrEmpty()) {
        var rSI = rSIs.Single();
        procSIs.Push(rSI);
        currSI = rSI;
        currTimesteps = currSI.Timesteps.OrderBy(ts => ts.TimeStepNumber.MajorNumber);
        computeTime = computeTime + (currTimesteps.Last().CreationTime - currTimesteps.First().CreationTime);
        rSIs = sessions.Where(sess => sess.ID.Equals(currSI.RestartedFrom));
    }

    Console.WriteLine($"Session - {sessName}: timesteps = {timesteps} (t_end = {physicalTime}); compute time = {computeTime.Days}:{computeTime.Hours}");
}
}

## TABLE 6

In [9]:
int index = 1;
foreach (var sess in evalSess) {
    var sessT = sess.Timesteps.WithoutSubSteps().OrderBy(ts => ts.TimeStepNumber.MajorNumber).First();
    int meshSize = sessT.Grid.NumberOfCells;
    int k = sessT.GetField("VelocityX").Basis.Degree;
    int NDOF = sessT.GetField("VelocityX").DOFLocal + sessT.GetField("VelocityY").DOFLocal + sessT.GetField("Pressure").DOFLocal;
    Console.WriteLine($"Setup {index}: No. of cells = {meshSize}, k = {k}, NDOF = {NDOF}");
    index++;
}

Error: Command cancelled.

## FIGURE 14 - Contact line position and velocity

In [9]:
var evalData = evalSess.ReadLogDataForMovingContactLine(); 

In [10]:
// plotting both contact line positions of the left interface
public static (Plot2Ddata pltCLpos, Plot2Ddata pltCLvel) GetCLpropertiesOverTime_Plot2Ddata(List<Plot2Ddata>[] data, string testcase) {

    Plot2Ddata pltCLpos =  new Plot2Ddata();
    pltCLpos.Xlabel = "Time";
    pltCLpos.Ylabel = "Position";

    Plot2Ddata pltCLvel =  new Plot2Ddata();
    pltCLvel.Xlabel = "Time";
    pltCLvel.Ylabel = "Velocity";


    int datGrpsCount = data[0].ElementAt(0).dataGroups.Count(); 
    int lcIdx = 1;
    for (int i = 0; i < datGrpsCount; i++) {

        var fmt = new PlotFormat();
        fmt.Style = Styles.Lines; 
        fmt.DashType = DashTypes.Solid;
        fmt.LineWidth = 2;
        fmt.LineColor = (LineColors)(lcIdx); //LineColors.Blue;

        // get index for both contact line positions of the left interface
        int idxCLposUpperLeft = -1;
        int idxCLposLowerLeft = -1;
        for (int cl = 0; cl < 4; cl++) {
            if (data[cl].ElementAt(0).dataGroups[i].Values[0] < 54.4) {
                if (data[cl].ElementAt(1).dataGroups[i].Values[0] > 6.8) {
                    idxCLposUpperLeft = cl;
                    continue;
                }
                if (data[cl].ElementAt(1).dataGroups[i].Values[0] < 6.8) {
                    idxCLposLowerLeft = cl;
                    continue;
                }
            }
        }

        if (idxCLposLowerLeft < 0 || idxCLposUpperLeft < 0) {
            for (int cl = 0; cl < 4; cl++) {
                Console.WriteLine($"CL pos check: cl={cl}, pos0={data[cl].ElementAt(0).dataGroups[i].Values[0]}, pos1={data[cl].ElementAt(1).dataGroups[i].Values[0]}");
            }
            throw new ApplicationException("Could not find indices for contact line positions of left interface.");
        }

        string[] nameData = data[0].ElementAt(0).dataGroups[i].Name.Split(new string[] { "_" }, StringSplitOptions.RemoveEmptyEntries);        
        if (nameData[1] == $"{testcase}") {

            double[] time = data[0].ElementAt(0).dataGroups[i].Abscissas;
            double[] upperPosX = data[idxCLposUpperLeft].ElementAt(0).dataGroups[i].Values;
            double[] lowerPosX = data[idxCLposLowerLeft].ElementAt(0).dataGroups[i].Values;
            double[] upperVelX = data[idxCLposUpperLeft].ElementAt(2).dataGroups[i].Values;
            double[] lowerVelX = data[idxCLposLowerLeft].ElementAt(2).dataGroups[i].Values;

            pltCLpos.AddDataGroup($"{nameData[3]}-{nameData[4]}", time, upperPosX, fmt);
            pltCLpos.AddDataGroup($"{nameData[3]}-{nameData[4]}", time, lowerPosX, fmt);
            
            pltCLvel.AddDataGroup($"{nameData[3]}-{nameData[4]}", time, upperVelX, fmt);
            pltCLvel.AddDataGroup($"{nameData[3]}-{nameData[4]}", time, lowerVelX, fmt);
            lcIdx++;
        }
    }

    pltCLpos.XrangeMin = 0.0;
    pltCLpos.XrangeMax = 160.0;
    pltCLpos.ShowLegend = true;
    pltCLpos.LegendAlignment = new string[] {"i", "c", "r"};

    pltCLvel.XrangeMin = 0.0;
    pltCLvel.XrangeMax = 160.0;
    pltCLvel.ShowLegend = false;

    return (pltCLpos, pltCLvel);
}

In [11]:
public static Plot2Ddata GetVelocityAtUpperWallForSymmetricCase(List<ISessionInfo> dataSess, int evalPeriod = 5) {

    Plot2Ddata plt =  new Plot2Ddata();
    plt.Xlabel = "Time";
    plt.Ylabel = "Velocity";

    double[] probe = new double[] { 54.4, 13.599 };
    int lcIdx = 1;
    foreach (var dataS in dataSess) {
        if (dataS.Name.Contains("_symmetric_")) {
            var timesteps = dataS.Timesteps.WithoutSubSteps().OrderBy(ts => ts.TimeStepNumber.MajorNumber).Every(evalPeriod);
            double[] time = new double[timesteps.Count()];
            double[] cpVeloc = new double[timesteps.Count()];

            for (int ts = 0; ts < timesteps.Count(); ts++) {
                time[ts] = timesteps.ElementAt(ts).PhysicalTime;
                DGField VelocityX = timesteps.ElementAt(ts).GetField("VelocityX");
                try {
                    double velXatProbe = VelocityX.ProbeAt(probe); 
                    cpVeloc[ts] = velXatProbe;   
                } catch {
                    cpVeloc[ts] = 0.0;
                }
            }

            var fmt = new PlotFormat();
            fmt.Style = Styles.Lines; 
            fmt.DashType = DashTypes.Solid;
            fmt.LineWidth = 2;
            fmt.LineColor = (LineColors)(lcIdx); //LineColors.Blue;

            string[] nameData = dataS.Name.Split(new string[] { "_" }, StringSplitOptions.RemoveEmptyEntries);
            plt.AddDataGroup($"{nameData[3]}-{nameData[4]}", time, cpVeloc, fmt);
            lcIdx++;
        }
    }

    plt.XrangeMin = 0.0;
    plt.XrangeMax = 160.0;
    plt.ShowLegend = false;

    return plt;

}

In [12]:
Plot2Ddata upperWallVelPlt = GetVelocityAtUpperWallForSymmetricCase(evalSess);

In [13]:
static string userName = System.Security.Principal.WindowsIdentity.GetCurrent().Name;
userName

FDY\smuda

In [14]:
public static Plot2Ddata GetReferenceData_Plot2Ddata() {

    Plot2Ddata plt =  new Plot2Ddata();
    plt.Xlabel = "Time";
    plt.Ylabel = "Velocity";

    // Add reference shapes
    string[] refDataS  = new string[] { "gerbeau_2009_fig11_velocity.csv", "gerbeau_2009_fig11_velocityWall.csv" };
    foreach (string dat in refDataS) {
        string path = (userName.Equals(@"FDY\jenkinsci")) ? dat : @"refData\" + dat;
        string[] lines = File.ReadAllLines(path);

        double[] time = new double[lines.Length];
        double[] valueDat = new double[lines.Length];

        for (int i = 0; i < lines.Length; i++) {
            time[i] = Convert.ToDouble(lines[i].Split(new string[] { "," }, StringSplitOptions.RemoveEmptyEntries)[0]);            
            valueDat[i] = Convert.ToDouble(lines[i].Split(new string[] { "," }, StringSplitOptions.RemoveEmptyEntries)[1]);
        }     

        var fmt = new PlotFormat();
        fmt.Style = Styles.Lines;
        fmt.Style      = Styles.Points;
        fmt.PointType  = PointTypes.Diamond;
        fmt.PointSize = 0.5;
        fmt.LineColor = (LineColors)(5);

        string[] nameData = dat.Split(new string[] { "_" }, StringSplitOptions.RemoveEmptyEntries);
        plt.AddDataGroup($"{nameData[0]}-{nameData[1]}", time, valueDat, fmt);
    }

    plt.XrangeMin = 0.0;
    plt.XrangeMax = 160.0;
    plt.ShowLegend = false;

    return plt;
}

In [15]:
Plot2Ddata velRefData = GetReferenceData_Plot2Ddata();

In [16]:
Plot2Ddata[,] Fig14 = new Plot2Ddata[2, 2];

(Plot2Ddata pltPos, Plot2Ddata pltVel) = GetCLpropertiesOverTime_Plot2Ddata(evalData, "symmetric");
Fig14[0, 0] = pltPos;
pltVel.AddDataGroup(upperWallVelPlt.dataGroups[0]);
pltVel.AddDataGroup(velRefData.dataGroups[0]);
pltVel.AddDataGroup(velRefData.dataGroups[1]);
Fig14[0, 1] = pltVel;

(pltPos, pltVel) = GetCLpropertiesOverTime_Plot2Ddata(evalData, "asymmetric");
Fig14[1, 0] = pltPos;
Fig14[1, 1] = pltVel;

Fig14.ToGnuplot().PlotSVG(xRes:1200,yRes:1000)

<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 22 
 
 
 
 
 23 
 
 
 
 
 24 
 
 
 
 
 25 
 
 
 
 
 26 
 
 
 
 
 27 
 
 
 
 
 28 
 
 
 
 
 29 
 
 
 
 
 30 
 
 
 
 
 31 
 
 
 
 
 32 
 
 
 
 
 0 
 
 
 
 
 20 
 
 
 
 
 40 
 
 
 
 
 60 
 
 
 
 
 80 
 
 
 
 
 100 
 
 
 
 
 120 
 
 
 
 
 140 
 
 
 
 
 160 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 Position 
 
 
 
 
 Time 
 
 
 
 
 k2-mesh2 
 
 
 
 
 k2-mesh2 
 
 
 
	<path stroke='rgb(255, 0, 0)' d='M382.5,179.3 L435.9,179.3 M69.5,231.0 L69.9,230.7 L70.2,230.3 L70.6,229.8 L71.0,229.3 L71.4,228.7
 L71.7,228.1 L72.1,227.5 L72.5,226.9 L72.8,226.3 L73.2,225.7 L73.6,225.1 L73.9,224.5 L74.3,223.8
 L74.7,223.2 L75.1,222.6 L75.4,221.9 L75.8,221.3 L76.2,220.6 L76.5,220.0 L76.9,219.3 L77.3,218.7
 L77.7,218.0 L78.0,217.4 L78.4,216.7 L78.8,216.0 L79.1,215.4 L79.5,214.7 L79.9,214.0 L80.3,213.4
 L80.6,212.7 L81.0,212.0 L81.4,211.4 L81.7,210.7 L82.1,210.0 L82.5,209.4 L82.8,208.7 L83.2,208.0
 L83.6,207.3 L84.0,206.6 L84.3,206.0 L84.7,205.3 L85.1,204.6 L85.4,203.9 L85.8,203.2 L86.2,202.5
 L86.6,201.9 L86.9,201.2 L87.3,200.5 L87.7,199.8 L88.0,199.1 L88.4,198.4 L88.8,197.7 L89.2,197.1
 L89.5,196.4 L89.9,195.7 L90.3,195.0 L90.6,194.3 L91.0,193.6 L91.4,193.0 L91.7,192.3 L92.1,191.6
 L92.5,190.9 L92.9,190.3 L93.2,189.6 L93.6,188.9 L94.0,188.3 L94.3,187.6 L94.7,187.0 L95.1,186.3
 L95.5,185.7 L95.8,185.0 L96.2,184.4 L96.6,183.7 L96.9,183.1 L97.3,182.5 L97.7,181.8 L98.1,181.2
 L98.4,180.6 L98.8,180.0 L99.2,179.4 L99.5,178.8 L99.9,178.2 L100.3,177.6 L100.6,177.0 L101.0,176.4
 L101.4,175.8 L101.8,175.2 L102.1,174.6 L102.5,174.1 L102.9,173.5 L103.2,172.9 L103.6,172.4 L104.0,171.8
 L104.4,171.2 L104.7,170.7 L105.1,170.1 L105.5,169.6 L105.8,169.0 L106.2,168.5 L106.6,168.0 L106.9,167.4
 L107.3,166.9 L107.7,166.4 L108.1,165.9 L108.4,165.4 L108.8,164.8 L109.2,164.3 L109.5,163.8 L109.9,163.3
 L110.3,162.8 L110.7,162.3 L111.0,161.8 L111.4,161.3 L111.8,160.8 L112.1,160.4 L112.5,159.9 L112.9,159.4
 L113.3,158.9 L113.6,158.5 L114.0,158.0 L114.4,157.5 L114.7,157.1 L115.1,156.6 L115.5,156.2 L115.8,155.7
 L116.2,155.3 L116.6,154.8 L117.0,154.4 L117.3,153.9 L117.7,153.5 L118.1,153.1 L118.4,152.6 L118.8,152.2
 L119.2,151.8 L119.6,151.4 L119.9,150.9 L120.3,150.5 L120.7,150.1 L121.0,149.7 L121.4,149.3 L121.8,148.9
 L122.2,148.5 L122.5,148.2 L122.9,147.8 L123.3,147.4 L123.6,147.0 L124.0,146.6 L124.4,146.2 L124.7,145.9
 L125.1,145.5 L125.5,145.1 L125.9,144.8 L126.2,144.4 L126.6,144.0 L127.0,143.7 L127.3,143.3 L127.7,143.0
 L128.1,142.6 L128.5,142.3 L128.8,141.9 L129.2,141.6 L129.6,141.3 L129.9,140.9 L130.3,140.6 L130.7,140.3
 L131.0,139.9 L131.4,139.6 L131.8,139.3 L132.2,139.0 L132.5,138.6 L132.9,138.3 L133.3,138.0 L133.6,137.7
 L134.0,137.4 L134.4,137.1 L134.8,136.7 L135.1,136.4 L135.5,136.1 L135.9,135.8 L136.2,135.5 L136.6,135.2
 L137.0,134.9 L137.4,134.6 L137.7,134.3 L138.1,134.0 L138.5,133.7 L138.8,133.4 L139.2,133.2 L139.6,132.9
 L139.9,132.6 L140.3,132.3 L140.7,132.0 L141.1,131.7 L141.4,131.5 L141.8,131.2 L142.2,130.9 L142.5,130.6
 L142.9,130.4 L143.3,130.1 L143.7,129.8 L144.0,129.6 L144.4,129.3 L144.8,129.0 L145.1,128.8 L145.5,128.5
 L145.9,128.2 L146.3,128.0 L146.6,127.7 L147.0,127.5 L147.4,127.2 L147.7,126.9 L148.1,126.7 L148.5,126.4
 L148.8,126.2 L149.2,125.9 L149.6,125.7 L150.0,125.4 L150.3,125.2 L150.7,125.0 L151.1,124.7 L151.4,124.5
 L151.8,124.2 L152.2,124.0 L152.6,123.8 L152.9,123.5 L153.3,123.3 L153.7,123.1 L154.0,122.8 L154.4,122.6
 L154.8,122.4 L155.2,122.1 L155.5,121.9 L155.9,121.7 L156.3,121.5 L156.6,121.2 L157.0,121.0 L157.4,120.8
 L157.7,120.5 L158.1,120.3 L158.5,120.1 L158.9,119.9 L159.2,119.7 L159.6,119.4 L160.0

In [17]:
public static void EvaluateTerminalTimeStep(Plot2Ddata pltDat, double[] lowerThrshldValues, double[] upperThrshldValues) {

    int numGrps = pltDat.dataGroups.Count();
    if (lowerThrshldValues.Length != numGrps || upperThrshldValues.Length != numGrps) {
        throw new ArgumentException("Threshold value arrays must match number of data groups.");
    }

    for (int i = 0; i < numGrps; i++) {
        var grp = pltDat.dataGroups[i];
        // last time step index
        int timeIdx  = grp.Abscissas.Length - 1;
        double time = grp.Abscissas[timeIdx];

        double valAtTime = grp.Values[timeIdx];
        Console.WriteLine($"Data group: {grp.Name}, value at time {time}: {valAtTime}");

        NUnit.Framework.Assert.IsTrue(valAtTime >= lowerThrshldValues[i], 
            $"Value {valAtTime} of data group {grp.Name} at time {time} is below lower threshold {lowerThrshldValues[i]}.");

        NUnit.Framework.Assert.IsTrue(valAtTime <= upperThrshldValues[i], 
            $"Value {valAtTime} of data group {grp.Name} at time {time} is above upper threshold {upperThrshldValues[i]}.");
    }
}

In [18]:
EvaluateTerminalTimeStep(Fig14[0,0], 
    new double[] { 31.1, 22.8, 31.3, 22.9, 31.2, 23.0 }, 
    new double[] { 31.6, 23.0, 31.4, 23.1, 31.4, 23.2 });

In [19]:
Plot2Ddata pltCheck =  new Plot2Ddata();
for (int i = 0; i < 7; i++) {   // removing reference data groups
    pltCheck.AddDataGroup(Fig14[0,1].dataGroups[i]);
}
EvaluateTerminalTimeStep(pltCheck, 
    new double[] { -0.003, -0.003, -0.002, -0.020, -0.002, -0.002, 0.20}, 
    new double[] { 0.003, 0.003, 0.002, 0.002, 0.002, 0.002, 0.22});

In [20]:
EvaluateTerminalTimeStep(Fig14[1,0], 
    new double[] { 28.2, 24.0, 28.2, 24.0, 28.2, 24.0 }, 
    new double[] { 28.3, 24.2, 28.3, 24.2, 28.3, 24.2 });

In [21]:
EvaluateTerminalTimeStep(Fig14[1,1], 
    new double[] { -0.003, -0.003, -0.002, -0.020, -0.002, -0.002 }, 
    new double[] { 0.003, 0.003, 0.002, 0.002, 0.002, 0.005 });

In [22]:
public static void CheckAgainstReferenceData(Plot2Ddata.XYvalues xyValues, Plot2Ddata.XYvalues refValues, double tolerance) {

    Dictionary<string, double> ret = ISessionInfoExtensions.ComputeErrorNormsForLogData(xyValues, refValues);

    // check l1 norm against tolerance
    NUnit.Framework.Assert.IsTrue(ret["l_1 error norm"] <= tolerance, 
        $"L1 norm {ret["l_1 error norm"]} exceeds tolerance {tolerance}.");
}

checking upper contact line velocity for the symmetric case

In [24]:
for (int i = 0; i < 3; i++) {
    Console.WriteLine($"Checking data group {Fig14[0,1].dataGroups[2*i].Name} against reference data.");
    CheckAgainstReferenceData(Fig14[0,1].dataGroups[2*i], velRefData.dataGroups[0], 1.1e-1);
}

checking upper wall velocity for $k=4$ and mesh0

In [25]:
CheckAgainstReferenceData(upperWallVelPlt.dataGroups[0], velRefData.dataGroups[1], 2e-3);

## FIGURE 16 - Interface shape

In [26]:
public static (double[] x, double[] y) GetTerminalInterfaceShape_XYvalues(ISessionInfo evalSess) {

    var terminalStep = evalSess.Timesteps.WithoutSubSteps().Where(ts => (ts.PhysicalTime - 160.0).Abs() <= 1e-9).Single();
    Console.WriteLine($"time step found at {terminalStep.PhysicalTime}");
    DGField phi = terminalStep.GetField("Phi");
    LevelSet LevSet = new LevelSet(phi.Basis, "LevelSet"); 
    LevSet.Acc(1.0, phi);           
    LevelSetTracker LsTrk = new LevelSetTracker((BoSSS.Foundation.Grid.Classic.GridData) phi.GridDat, CutCellQuadratureMethod.Saye, 1, new string[] { "A", "B" }, LevSet);
    LsTrk.UpdateTracker(terminalStep.PhysicalTime);

    MultidimensionalArray interP = BoSSS.Solution.XNSECommon.XNSEUtils.GetInterfacePoints(LsTrk, LevSet, quadRuleOrderForNodeSet:10);
    double[] x = interP.ExtractSubArrayShallow(new int[] { -1, 0 }).To1DArray();
    double[] y = interP.ExtractSubArrayShallow(new int[] { -1, 1 }).To1DArray();

    List<double> x_left = new List<double>();
    List<double> y_left = new List<double>();
    int numP = x.Length;
    for(int i = 0; i < numP; i++) {
        if(x[i] < 54.4) {
            x_left.Add(x[i] - 27.2);
            y_left.Add(y[i]);
        }
    }

    return (x_left.ToArray(), y_left.ToArray());
    
}

In [27]:
public static Plot2Ddata GetShapeComparison_Plot2Ddata(List<ISessionInfo> dataSess) {

    Plot2Ddata datShape = new Plot2Ddata();

    int lcIdx = 4;
    // Add reference shapes
    string[] refDataS  = new string[] { "Qian_2006_fig20_symmetric.csv", "gerbeau_2009_fig12_symmetric.csv", "Qian_2006_fig20_asymmetric.csv", "gerbeau_2009_fig12_asymmetric.csv" };
    foreach (string dat in refDataS) {
        string path = (userName.Equals(@"FDY\jenkinsci")) ? dat : @"refData\" + dat;
        string[] lines = File.ReadAllLines(path);
        double[] x = new double[lines.Length];
        double[] y = new double[lines.Length];

        for (int i = 0; i < lines.Length; i++) {
            x[i] = Convert.ToDouble(lines[i].Split(new string[] { "," }, StringSplitOptions.RemoveEmptyEntries)[0]);            
            y[i] = Convert.ToDouble(lines[i].Split(new string[] { "," }, StringSplitOptions.RemoveEmptyEntries)[1]);      
        } 

        var fmt = new PlotFormat();
        fmt.Style      = Styles.Points;
        fmt.PointType  = PointTypes.Diamond;
        fmt.PointSize = 0.5;
        fmt.LineColor = (LineColors)(lcIdx);

        string[] nameData = dat.Split(new string[] { "_" }, StringSplitOptions.RemoveEmptyEntries);
        datShape.AddDataGroup($"{nameData[0]}-{nameData[1]}", x, y, fmt);

        lcIdx++;
        if (lcIdx > 5)
            lcIdx = 4;
    }

    lcIdx = 1;
    //for(int i = 0; i < study.Length; i++) {
        foreach (var dataS in dataSess) {
            //if (dataS.Name.Contains(study[i])) {
                var shape = GetTerminalInterfaceShape_XYvalues(dataS);

                var fmt = new PlotFormat();
                fmt.Style      = Styles.Points;
                fmt.PointType  = PointTypes.Circle;
                fmt.PointSize = 0.5;
                fmt.LineColor = (LineColors)(lcIdx);

                string[] nameData = dataS.Name.Split(new string[] { "_" }, StringSplitOptions.RemoveEmptyEntries);
                datShape.AddDataGroup($"{nameData[3]}-{nameData[4]}", shape.x, shape.y, fmt);
                
                lcIdx++;
                if (lcIdx > 3)
                    lcIdx = 1;
            //}
        }
    //}

    datShape.YrangeMin = 0.0;
    datShape.YrangeMax = 13.6;
    datShape.LegendAlignment = new string[] {"o", "b", "r"};

    return datShape;
}

In [28]:
Plot2Ddata shapePlt = GetShapeComparison_Plot2Ddata(evalSess);

shapePlt.ToGnuplot().PlotSVG(xRes:800,yRes:500)

<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 2 
 
 
 
 
 4 
 
 
 
 
 6 
 
 
 
 
 8 
 
 
 
 
 10 
 
 
 
 
 12 
 
 
 
 
 -5 
 
 
 
 
 -4 
 
 
 
 
 -3 
 
 
 
 
 -2 
 
 
 
 
 -1 
 
 
 
 
 0 
 
 
 
 
 1 
 
 
 
 
 2 
 
 
 
 
 3 
 
 
 
 
 4 
 
 
 
 
 5 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 Qian-2006 
 
 
 Qian-2006 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 gerbeau-2009 
 
 
 gerbeau-2009 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 Qian-2006 
 
 
 Qian-2006 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 gerbeau-2009 
 
 
 gerbeau-2009 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 k2-mesh2 
 
 
 k2-mesh2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 k3-mesh1 
 
 
 k3-mesh1 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 

In [29]:
public static void CheckInterfaceShapeAgainstReferenceData(Plot2Ddata.XYvalues xyValues, Plot2Ddata.XYvalues refValues, double tolerance) {

    double distError = 0.0;
    for (int i = 0; i < refValues.Values.Length; i++) {
        double minDist = double.MaxValue;
        for (int j = 0; j < xyValues.Values.Length; j++) {
            double dist = Math.Sqrt( (refValues.Abscissas[i] - xyValues.Abscissas[j]).Pow2() + (refValues.Values[i] - xyValues.Values[j]).Pow2() );
            if (dist < minDist)
                minDist = dist;
        }
        distError += minDist;
    }
    distError /= refValues.Values.Length;

    // check l1 norm against tolerance
    NUnit.Framework.Assert.IsTrue(distError <= tolerance, 
        $"Anbsolute distance error {distError} of interface shape exceeds tolerance {tolerance}.");
}

checking the terminal shape for the symmetric case against Qian(2006)

In [30]:
for (int i = 0; i < 3; i++) {
    Console.WriteLine($"Checking data group {shapePlt.dataGroups[4+i].Name} against reference data.");
    CheckInterfaceShapeAgainstReferenceData(shapePlt.dataGroups[4+i], shapePlt.dataGroups[0], 3e-1);
}

checking the terminal shape for the asymmetric case against Qian(2006)

In [31]:
for (int i = 0; i < 3; i++) {
    Console.WriteLine($"Checking data group {shapePlt.dataGroups[7+i].Name} against reference data.");
    CheckInterfaceShapeAgainstReferenceData(shapePlt.dataGroups[7+i], shapePlt.dataGroups[2], 5e-2);
}

checking the terminal shape for the symmetric case against Gerbeau(2009)

In [32]:
for (int i = 0; i < 3; i++) {
    Console.WriteLine($"Checking data group {shapePlt.dataGroups[4+i].Name} against reference data.");
    CheckInterfaceShapeAgainstReferenceData(shapePlt.dataGroups[4+i], shapePlt.dataGroups[1], 2e-1);
}

checking the terminal shape for the asymmetric case against Gerbeau(2009)

In [33]:
for (int i = 0; i < 3; i++) {
    Console.WriteLine($"Checking data group {shapePlt.dataGroups[7+i].Name} against reference data.");
    CheckInterfaceShapeAgainstReferenceData(shapePlt.dataGroups[7+i], shapePlt.dataGroups[3], 2e-1);
}